In [0]:
# Imports

from pyspark.sql.functions import col, trim, to_date, expr, when, current_timestamp
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import col

In [0]:
# Ler a Bronze V2
df_bronze = spark.table("workspace.drs_bronze.socio_demografico_ibge")
display(df_bronze)

In [0]:
# Padronizando os nomes das colunas
df_sociodemografico = df_bronze \
    .withColumnRenamed("municpio", "city_name") \
    .withColumnRenamed("uf", "state_abbreviation") \
    .withColumnRenamed("cdigo", "city_code") \
    .withColumnRenamed("gentlico", "demonym") \
    .withColumnRenamed("prefeito_2021", "mayor_2021") \
    .withColumnRenamed("rea_territorial__km_2022", "territorial_area_km2_2022") \
    .withColumnRenamed("populao_residente__pessoas_2022", "resident_population_2022") \
    .withColumnRenamed("densidade_demogrfica__habkm_2022", "population_density_hab_km2_2022") \
    .withColumnRenamed("escolarizao_span6_a_14_anosspan___2010", "school_enrollment_6_14_years_2010") \
    .withColumnRenamed("idhm_spanndice_de_desenvolvimento_humano_municipalspan_2010", "hdi_2010") \
    .withColumnRenamed("mortalidade_infantil__bitos_por_mil_nascidos_vivos_2020", "infant_mortality_per_1000_births_2020") \
    .withColumnRenamed("receitas_realizadas__r_1000_2017", "realized_revenues_brl_1000_2017") \
    .withColumnRenamed("despesas_empenhadas__r_1000_2017", "committed_expenses_brl_1000_2017") \
    .withColumnRenamed("pib_per_capita__r_2020", "gdp_per_capita_brl_2020") \

print(f"Total registros: {df_sociodemografico.count()}")
display(df_sociodemografico)


In [0]:
# Transformar/Convert colunas para o padarão para depois salvar a Silver V2

from pyspark.sql.functions import col, trim, to_date, expr, when, regexp_replace

df_silver = (
    df_sociodemografico
    .withColumn("territorial_area_km2_2022", regexp_replace(col("territorial_area_km2_2022"), ",", ".").cast("decimal(18,3)"))
    .withColumn("population_density_hab_km2_2022", regexp_replace(col("population_density_hab_km2_2022"), ",", ".").cast("decimal(18,3)"))
    .withColumn("school_enrollment_6_14_years_2010", regexp_replace(col("school_enrollment_6_14_years_2010"), ",", ".").cast("decimal(18,3)"))
    .withColumn("hdi_2010", regexp_replace(col("hdi_2010"), ",", ".").cast("decimal(18,3)"))
    .withColumn("infant_mortality_per_1000_births_2020", regexp_replace(col("infant_mortality_per_1000_births_2020"), ",", ".").cast("decimal(18,3)"))
    .withColumn("realized_revenues_brl_1000_2017", regexp_replace(col("realized_revenues_brl_1000_2017"), ",", ".").cast("decimal(18,3)"))
    .withColumn("committed_expenses_brl_1000_2017", regexp_replace(col("committed_expenses_brl_1000_2017"), ",", ".").cast("decimal(18,3)"))
    .withColumn("gdp_per_capita_brl_2020", regexp_replace(col("gdp_per_capita_brl_2020"), ",", ".").cast("decimal(18,3)"))
)


In [0]:
# criando o campo source_file
from pyspark.sql.functions import lit

df = df_silver.withColumn("source_file", lit("socio_demografico_ibge.csv"))

In [0]:
# criando o campo ingestion_timestamp
from pyspark.sql.functions import current_timestamp

df_silver = df.withColumn("ingestion_timestamp", current_timestamp())

In [0]:
# Criando o campo silver_processed_timestamp em df_silver_v2
from pyspark.sql.functions import current_timestamp

df_socio_demografico = df_silver \
.withColumn("silver_processed_timestamp", current_timestamp()) \
.withColumn("created_at", current_timestamp()) \
.withColumn("updated_at", current_timestamp())

In [0]:
# # Grantindo a deduplicação por chave de negócio

# window_spec = Window.partitionBy(
#     "generation_date",
#     "section_number",
#     "office_code",
# ).orderBy(
#     col("ingestion_timestamp").desc()
# )

# df_eleicoes = (
#     df_eleicoes
#     .withColumn("row_num", row_number().over(window_spec))
#     .filter(col("row_num") == 1)
#     .drop("row_num")
# )

In [0]:
# Validando shema da dataframe v2
df_socio_demografico.printSchema()

In [0]:
# Salvando a tabela Silver em uma staging table

df_socio_demografico.write \
.format("delta") \
.mode("overwrite") \
.option("overwriteSchema", "true") \
.saveAsTable("workspace.drs_silver.socio_demografico_staging")

In [0]:

# Craindo tabela final se não existir

spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.drs_silver.socio_demografico
AS
SELECT * FROM workspace.drs_silver.socio_demografico_staging WHERE 1 = 0
""")

In [0]:
# Validar schema da staging

spark.table("workspace.drs_silver.socio_demografico_staging").printSchema()

In [0]:
# Salvando tabela drs_silver_v2 com MERGE


spark.sql("""
MERGE INTO workspace.drs_silver.socio_demografico AS target
USING workspace.drs_silver.socio_demografico_staging AS source

ON target.city_code = source.city_code

WHEN MATCHED THEN
  UPDATE SET
    target.city_name = source.city_name,
    target.state_abbreviation = source.state_abbreviation,
    target.demonym = source.demonym,
    target.mayor_2021 = source.mayor_2021,
    target.territorial_area_km2_2022 = source.territorial_area_km2_2022,
    target.resident_population_2022 = source.resident_population_2022,
    target.population_density_hab_km2_2022 = source.population_density_hab_km2_2022,
    target.school_enrollment_6_14_years_2010 = source.school_enrollment_6_14_years_2010,
    target.hdi_2010 = source.hdi_2010,
    target.infant_mortality_per_1000_births_2020 = source.infant_mortality_per_1000_births_2020,
    target.realized_revenues_brl_1000_2017 = source.realized_revenues_brl_1000_2017,
    target.committed_expenses_brl_1000_2017 = source.committed_expenses_brl_1000_2017,
    target.gdp_per_capita_brl_2020 = source.gdp_per_capita_brl_2020,
    target.ingestion_timestamp = source.ingestion_timestamp,
    target.source_file = source.source_file,
    target.silver_processed_timestamp = source.silver_processed_timestamp,
    target.updated_at = current_timestamp()

WHEN NOT MATCHED THEN
  INSERT (
    city_name,
    state_abbreviation,
    city_code,
    demonym,
    mayor_2021,
    territorial_area_km2_2022,
    resident_population_2022,
    population_density_hab_km2_2022,
    school_enrollment_6_14_years_2010,
    hdi_2010,
    infant_mortality_per_1000_births_2020,
    realized_revenues_brl_1000_2017,
    committed_expenses_brl_1000_2017,
    gdp_per_capita_brl_2020,
    ingestion_timestamp,
    source_file,
    silver_processed_timestamp,
    created_at,
    updated_at
  )
  VALUES (
    source.city_name,
    source.state_abbreviation,
    source.city_code,
    source.demonym,
    source.mayor_2021,
    source.territorial_area_km2_2022,
    source.resident_population_2022,
    source.population_density_hab_km2_2022,
    source.school_enrollment_6_14_years_2010,
    source.hdi_2010,
    source.infant_mortality_per_1000_births_2020,
    source.realized_revenues_brl_1000_2017,
    source.committed_expenses_brl_1000_2017,
    source.gdp_per_capita_brl_2020,
    source.ingestion_timestamp,
    source.source_file,
    source.silver_processed_timestamp,
    current_timestamp(),
    current_timestamp()
  )
""")

In [0]:
# Validando a tabela drs_silver_votacao_secao_2024_go

spark.sql("""
SELECT
COUNT(*) AS total_rows
FROM workspace.drs_silver.socio_demografico
""").display()

In [0]:
# # Verificar duplicidade pela chave do merge

# spark.sql("""
# SELECT
#     generation_date,
#     section_number,
#     office_code,
#     COUNT(*) AS qtd
# FROM workspace.drs_silver.votacao_secao_2024_go
# GROUP BY generation_date, section_number, office_code 
# HAVING COUNT(*) > 1
# ORDER BY qtd DESC
# """).display()

In [0]:
# # Data Quality checks da Silver V2
# dq = spark.sql("""
# SELECT
#     SUM(CASE WHEN vote_count < 0 THEN 1 ELSE 0 END) AS vote_count,
#     SUM(CASE WHEN generation_date IS NULL THEN 1 ELSE 0 END) AS generation_date,
#     SUM(CASE WHEN election_date IS NULL THEN 1 ELSE 0 END) AS election_date,
#     SUM(CASE WHEN ballot_number is NULL THEN 1 ELSE 0 END) AS ballot_number
# FROM workspace.drs_silver.votacao_secao_2024_go
# """)

# display(dq)

In [0]:
# validação quarentena
spark.sql("""
SELECT COUNT(*) AS total_invalid
FROM workspace.drs_silver.socio_demografico
""").display()

In [0]:
# # Falhar notebook se houver erro crítico de qualidade
# dq_row = dq.collect()[0]

# if (
#     dq_row["vote_count"] > 0 or
#     dq_row["ballot_number"] > 0 
# ):
#     raise Exception("Data Quality check failed in workspace.drs_silver.sales_v2")
